# LeapQuant KDA 教程 (4/5): 窗口扫描 —— flush 间隔 vs 漂移

LeapQuant 只在**窗口边界**做 int8 快照 (窗口内 fp32 递推, 不积累量化误差)。
flush 越频繁 → 省的显存恢复越快, 但快照误差注入越多。本册扫 $W \in \{8, 16, 32, 64, \infty\}$,
并回答一个 KDA 特有问题: **不同 $g_j$ 的列, flush 误差一样吗?**

In [1]:
import sys, math
import torch

sys.path.insert(0, "/workspace/y00924007/vllm-ascend")  # 开发仓 (kernel / 量化器所在)
torch.npu.set_device(0)
DEV = torch.device("npu:0")
torch.manual_seed(7)
print("device:", torch.npu.get_device_name(0))


from vllm_ascend.quantization.methods.mamba_state.leapquant.state import leap_window_flush

H, V, K = 8, 128, 128
T = 64


def trajectory(per_k, window, seed=11):
    """roll T 步, 每 window 步 leap_window_flush; 返回 (输出, 是否有 flush)."""
    torch.manual_seed(seed)
    g_vec = -0.1 - 0.6 * torch.rand(H, K, device=DEV)
    g_sca = g_vec.mean(dim=1)
    S = torch.zeros(H, V, K, device=DEV)
    outs = []
    for t in range(T):
        k = torch.nn.functional.normalize(torch.randn(H, K, device=DEV), dim=-1)
        q = torch.nn.functional.normalize(torch.randn(H, K, device=DEV), dim=-1)
        v = torch.randn(H, V, device=DEV)
        beta = torch.rand(H, 1, device=DEV) * 0.5 + 0.5
        g = g_vec if per_k else g_sca[:, None]
        S = S * torch.exp(g)[:, None, :]
        sk = torch.einsum("hvk,hk->hv", S, k)
        S = S + torch.einsum("hv,hk->hvk", (v - sk) * beta, k)
        outs.append(torch.einsum("hvk,hk->hv", S, q))
        if window and (t + 1) % window == 0:
            S = leap_window_flush(S, compensator_tokens=4)
    return torch.stack(outs)

/usr/local/python3.11.10/lib/python3.11/site-packages/torch_npu/_init/core/optional_features.py:20: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. Do not set it to 1 to avoid unexpected errors.
  warnings.warn(


device: Ascend910_9382


/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:13: FutureWarning: triton.language.extra.cann.extension.atan2 is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.atan2 instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.atan2")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:35: FutureWarning: triton.language.extra.cann.extension.isfinited is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.isfinited instead.
  @_deprecated(replacement="triton.language.extra.cann.libdevice.isfinited")
/usr/local/python3.11.10/lib/python3.11/site-packages/triton/language/extra/cann/extension/math_ops.py:49: FutureWarning: triton.language.extra.cann.extension.finitef is deprecated and will be removed in the next release; use triton.language.extra.cann.libdevice.finitef instead.
  @_deprec

INFO 11-11 16:22:44 [__init__.py:52] Available plugins for group vllm.platform_plugins:


INFO 11-11 16:22:44 [__init__.py:54] - ascend -> vllm_ascend:register


INFO 11-11 16:22:44 [__init__.py:57] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 11-11 16:22:44 [__init__.py:272] Platform plugin ascend is activated


INFO 11-11 16:22:45 [platform.py:85] Breakable CUDAGraph on Ascend is opt-in; using VLLM_USE_BREAKABLE_CUDAGRAPH=0.


/usr/local/python3.11.10/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 窗口扫描: KDA vs GDN

In [2]:
print("  window      KDA 漂移        GDN 漂移")
for W in [8, 16, 32, 64, None]:
    rel = {}
    for per_k, tag in [(True, "KDA"), (False, "GDN")]:
        ref = trajectory(per_k, None)
        run = trajectory(per_k, W)
        rel[tag] = ((run - ref).norm() / ref.norm()).item()
    ws = f"{W:6d}" if W else "   无(参考)"
    print(f"{ws}     {rel['KDA']:.4f}         {rel['GDN']:.4f}")

  window      KDA 漂移        GDN 漂移


[W1111 16:22:51.991789443 NPUCachingAllocator.cpp:204] Warning: The current CANN and HDK(driver) versions require processing for 32 padding size, with memory allocation. (function operator())


/workspace/y00924007/vllm-ascend/vllm_ascend/quantization/methods/mamba_state/leapquant/state.py:75: UserWarning: Cannot create tensor with interal format while allow_internel_format=False, tensor will be created with base format. (Triggered internally at ../torch_npu/csrc/aten/common/TensorFactories.cpp:437.)
  return q0.to(device=device, dtype=dtype).contiguous()


     8     0.0010         0.0004


    16     0.0007         0.0002


    32     0.0004         0.0001


    64     0.0000         0.0000


   无(参考)     0.0000         0.0000


## 2. KDA 特有: flush 误差的列分布

冲销后的残差经过逐列/逐行双尺度 int8 —— 理论上每列误差应该大致均匀;
但列范数分层 (教程 1) 意味着**相对**误差在快遗忘的列上更大。看一眼:

In [3]:
torch.manual_seed(11)
g_vec = -0.1 - 0.6 * torch.rand(H, K, device=DEV)
S = torch.zeros(H, V, K, device=DEV)
for _ in range(16):
    k = torch.nn.functional.normalize(torch.randn(H, K, device=DEV), dim=-1)
    v = torch.randn(H, V, device=DEV)
    beta = torch.rand(H, 1, device=DEV) * 0.5 + 0.5
    S = S * torch.exp(g_vec)[:, None, :]
    sk = torch.einsum("hvk,hk->hv", S, k)
    S = S + torch.einsum("hv,hk->hvk", (v - sk) * beta, k)

S_hat = leap_window_flush(S.clone(), compensator_tokens=4)
col_err = (S_hat - S).norm(dim=(0, 1))                         # 每列误差 [K]
col_norm = S.norm(dim=(0, 1))                                  # 每列范数 [K]
mean_g = g_vec.mean(dim=0)                                     # 每列平均门 [K]

order = mean_g.argsort()                                       # 门最负 -> 最不负
print("门最负的 6 列:   平均g     列范数   相对误差")
for j in order[:6].tolist():
    print(f"  列 {j:3d}:   {mean_g[j]:6.3f}   {col_norm[j]:7.2f}   {col_err[j] / col_norm[j]:.4f}")
print("门最不负的 6 列:")
for j in order[-6:].tolist():
    print(f"  列 {j:3d}:   {mean_g[j]:6.3f}   {col_norm[j]:7.2f}   {col_err[j] / col_norm[j]:.4f}")
print("\n--> 绝对误差几乎均匀 (双尺度 int8 的功劳), 快遗忘列只是范数小")

门最负的 6 列:   平均g     列范数   相对误差
  列  64:   -0.534      2.67   0.0023
  列  75:   -0.506      2.69   0.0022
  列  92:   -0.503      3.25   0.0022
  列 113:   -0.500      4.61   0.0014
  列  93:   -0.493      3.31   0.0017
  列   3:   -0.493      2.90   0.0020
门最不负的 6 列:
  列  23:   -0.303      3.84   0.0024
  列  29:   -0.298      3.46   0.0025
  列  48:   -0.294      2.95   0.0029
  列  72:   -0.292      3.27   0.0027
  列  58:   -0.290      4.08   0.0021
  列  39:   -0.286      3.48   0.0027

--> 绝对误差几乎均匀 (双尺度 int8 的功劳), 快遗忘列只是范数小


## 小结 (KDA 教程 4)

- flush 间隔的权衡与 GDN 一致: $W$ 越小漂移越大, $W{=}16$ 时漂移仍在小几个百分点内;
- KDA 与 GDN 在同一 $W$ 下漂移量级相同 —— 窗口机制对门的结构不敏感;
- 双尺度 int8 让 flush 的**绝对**误差按列接近均匀; 快遗忘列范数小, 相对误差天然大, 但对输出贡献也小。